# 🤖 Local Enterprise AI: Google Colab Master Setup
--- 
### 🌟 Giới thiệu dự án
Quy trình này cho phép bạn chạy hệ thống **Phân tích dữ liệu 83.6GB Amazon Reviews** trên hạ tầng Cloud của Google. 

**Tại sao dùng Colab?**
1. **Tận dụng GPU**: Dùng các card đồ họa mạnh (T4, A100) của Google để chạy mô hình AI (Llama 3) nhanh hơn CPU Local.
2. **Tiết kiệm tài nguyên máy tính**: Cho phép máy cá nhân của bạn nghỉ ngơi trong khi Cloud thực hiện các tác vụ nặng (Tải và Làm sạch dữ liệu).
3. **Tốc độ mạng**: Colab có tốc độ tải từ Hugging Face cực nhanh (~100MB/s).

### 🛠️ Bước 1: Kết nối Google Drive (Persistent Storage)
**Tác dụng**: Vì Colab sẽ xóa toàn bộ dữ liệu khi bạn tắt trình duyệt, chúng ta cần nạp Google Drive để lưu trữ 80GB dữ liệu một cách vĩnh viễn.

**Cách chạy**: Nhấn nút Play. Một cửa sổ sẽ hiện ra yêu cầu bạn cấp quyền truy cập Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Di chuyển vào thư mục dự án trên Drive của bạn
import os
project_path = "/content/drive/MyDrive/He_quan_tri"
if not os.path.exists(project_path):
    os.makedirs(project_path)

%cd $project_path

### 📦 Bước 2: Cài đặt "Cấu hình Kỹ thuật" (Infrastructure)
**Tác dụng**: Cài đặt các thư viện Python chuyên dụng như `hf_transfer` để tải dữ liệu tốc độ cao và `pandas` để xử lý dữ liệu lớn.

**Tại sao cần?**: Nếu không có bước này, Python sẽ không hiểu được các script chuyên sâu mà chúng ta đã viết.

In [ ]:
!pip install -r requirements.txt
!pip install hf_transfer

# Kích hoạt bộ tăng tốc tải dữ liệu
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

### 🧠 Bước 3: Khởi động Bộ não AI (Ollama GPU)
**Tác dụng**: Cài đặt và khởi chạy Ollama ngay trong Colab. 

**Tại sao cần?**: Bước này biến Colab thành một "AI Server". Chúng ta sẽ tải mô hình **Llama 3** về để hỗ trợ việc phân tích ngữ nghĩa và dịch câu hỏi tự nhiên thành SQL.

In [ ]:
print("📥 Đang cài đặt Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print("🚀 Khởi chạy Ollama Server (Chạy ngầm)... Chiếm dụng GPU...")
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=stderr=subprocess.PIPE)
time.sleep(10) # Chờ server khởi động

print("🤖 Đang tải mô hình Llama 3 (8B) - Trí tuệ nhân tạo chính...")
!ollama pull llama3
print("📝 Đang tải mô hình bge-m3 - Phục vụ Vector Search...")
!ollama pull bge-m3

### 📥 Bước 4: Tải và Làm sạch dữ liệu Enterprise (83.6GB)
**Tác dụng**: Chạy chuỗi script tự động hóa: Tải từ Hugging Face -> Sắp xếp folder -> Làm sạch CSV.

**Cách chạy**: Bước này có thể mất từ 30-60 phút tùy thuộc vào việc bạn chọn tải toàn bộ 80GB hay chỉ 1 phần. 

**Lưu ý**: Dữ liệu sẽ được nén và làm sạch ở chế độ **Safe-Mode** để không gây tràn bộ nhớ Drive.

In [ ]:
# 1. Tải dữ liệu (Electronics, Home, Tools)
!python3 python/enterprise_download_80gb.py

# 2. Sắp xếp sơ đồ thư mục Pro
!python3 python/organize_data.py

# 3. Làm sạch và chuẩn hóa CSV (Engineering)
!python3 python/process_for_external.py

### 🌉 Bước 5: Đóng gói và Chuyển giao (Bridge)
**Tác dụng**: Sau khi dữ liệu đã sạch (ở dạng CSV), chúng ta sẽ đóng gói chúng lại để bạn có thể tải về máy Local nạp vào Oracle Database.

**Tại sao cần?**: Google Colab không chạy được Oracle Database tốt, nên chúng ta dùng nó như một kho "Sơ chế dữ liệu" trước khi đưa vào kho lưu trữ chính.

In [ ]:
from python.colab_bridge import prepare_remote_ingest
prepare_remote_ingest()

print("✅ Xong! Hãy tải file 'colab_ingest_payload.zip' từ Drive về máy cá nhân của bạn.")